In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


## Day 11 - Window Functions & SCDs


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
df_fact = spark.table('fact_orders')
df_prod = spark.table('dim_product')
df_cust = spark.table('dim_customer')
print('Tables loaded')

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 3, Finished, Available, Finished, False)

Tables loaded


In [3]:

product_sales = df_fact \
.join(df_prod.select('product_key','product_name','category'), on='product_key',
how='left') \
.groupBy('product_key','product_name','category') \
.agg(F.round(F.sum('sales'),2).alias('total_sales'))
# Step 2: define a window partitioned by category, ordered by sales descending
window_cat = Window.partitionBy('category').orderBy(F.desc('total_sales'))
# Step 3: add rank column
product_sales = product_sales.withColumn('rank_in_category',
F.rank().over(window_cat))
# Show top 3 per category
product_sales.filter(F.col('rank_in_category') <= 3) \
.orderBy('category', 'rank_in_category') \
.show(20, truncate=False)

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 5, Finished, Available, Finished, False)

+-----------+---------------------------------------------------------------------------+---------------+-----------+----------------+
|product_key|product_name                                                               |category       |total_sales|rank_in_category|
+-----------+---------------------------------------------------------------------------+---------------+-----------+----------------+
|717        |HON 5400 Series Task Chairs for Big and Tall                               |Furniture      |21870.58   |1               |
|1210       |Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish              |Furniture      |15610.97   |2               |
|1226       |Bretford Rectangular Conference Table Tops                                 |Furniture      |13565.57   |3               |
|1291       |Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind|Office Supplies|27453.38   |1               |
|1178       |GBC DocuBind TL300 Electric Binding System

In [4]:
from pyspark.sql.functions import sum as spark_sum
# Aggregate to month level first
df_date = spark.table('dim_date')
monthly = df_fact \
.join(df_date.select('date_key','year','month'), on='date_key', how='left') \
.groupBy('year','month') \
.agg(F.round(F.sum('sales'),2).alias('monthly_sales')) \
.orderBy('year','month')
# Window: ordered by year then month, rows from beginning to current
window_time = Window.partitionBy('year').orderBy('month') \
.rowsBetween(Window.unboundedPreceding, Window.currentRow)
monthly = monthly.withColumn(
'running_total_ytd',
F.round(spark_sum('monthly_sales').over(window_time), 2)
)
monthly.show(24)

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 7, Finished, Available, Finished, False)

+----+-----+-------------+-----------------+
|year|month|monthly_sales|running_total_ytd|
+----+-----+-------------+-----------------+
|2023|    1|     15185.28|         15185.28|
|2023|    2|      4232.61|         19417.89|
|2023|    3|     58117.97|         77535.86|
|2023|    4|     28766.59|        106302.45|
|2023|    5|     27219.91|        133522.36|
|2023|    6|      34598.9|        168121.26|
|2023|    7|     33952.05|        202073.31|
|2023|    8|     30202.75|        232276.06|
|2023|    9|     83712.04|         315988.1|
|2023|   10|     32544.83|        348532.93|
|2023|   11|     85854.96|        434387.89|
|2023|   12|     75906.49|        510294.38|
|2024|    1|     18565.18|         18565.18|
|2024|    2|     11986.31|         30551.49|
|2024|    3|     42283.07|         72834.56|
|2024|    4|     34838.42|        107672.98|
|2024|    5|     29757.54|        137430.52|
|2024|    6|     24679.79|        162110.31|
|2024|    7|     29530.39|         191640.7|
|2024|    

In [7]:
product_sales.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_product_rankings")

print(f"gold_product_rankings written: {product_sales.count()} rows")

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 11, Finished, Available, Finished, False)

gold_product_rankings written: 1894 rows


In [9]:
from pyspark.sql.functions import lit, current_date
from pyspark.sql.types import DateType, BooleanType

# Read current dim_customer
df_dim = spark.table("dim_customer")

# Add SCD Type 2 columns
df_dim_scd = (
    df_dim
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast(DateType()))
    .withColumn("is_current", lit(True).cast(BooleanType()))
)

# Overwrite table with updated schema
df_dim_scd.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("dim_customer")

print(f"SCD columns added. Rows: {df_dim_scd.count()}")

df_dim_scd.show(5)

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 16, Finished, Available, Finished, False)

SCD columns added. Rows: 804
+------------+-----------+--------------+-----------+----------+--------+----------+
|customer_key|customer_id| customer_name|    segment|start_date|end_date|is_current|
+------------+-----------+--------------+-----------+----------+--------+----------+
|           0|   LS-17200|  Luke Schmidt|  Corporate|2026-07-02|    NULL|      true|
|           1|   PN-18775|Parhena Norris|Home Office|2026-07-02|    NULL|      true|
|           2|   DP-13105|  Dave Poirier|  Corporate|2026-07-02|    NULL|      true|
|           3|   GH-14665|   Greg Hansen|   Consumer|2026-07-02|    NULL|      true|
|           4|   MG-17875| Michael Grace|Home Office|2026-07-02|    NULL|      true|
+------------+-----------+--------------+-----------+----------+--------+----------+
only showing top 5 rows



In [11]:
# Cell S2 -- Simulate segment changes for 2 customers

# Pick any two customer_ids from dim_customer
sample_ids = spark.table("dim_customer").limit(2).select("customer_id").collect()

id1 = sample_ids[0]["customer_id"]
id2 = sample_ids[1]["customer_id"]

print(f"Simulating segment change for: {id1} and {id2}")

from pyspark.sql import Row

# New incoming data: these customers now have a different segment
changes = spark.createDataFrame([
    Row(customer_id=id1, customer_name="-- keep existing --", new_segment="Corporate"),
    Row(customer_id=id2, customer_name="-- keep existing --", new_segment="Home Office")
])

changes.show()

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 22, Finished, Available, Finished, False)

Simulating segment change for: LS-17200 and PN-18775
+-----------+-------------------+-----------+
|customer_id|      customer_name|new_segment|
+-----------+-------------------+-----------+
|   LS-17200|-- keep existing --|  Corporate|
|   PN-18775|-- keep existing --|Home Office|
+-----------+-------------------+-----------+



In [13]:
# Cell S3 -- Apply SCD Type 2: close old rows + insert new rows

from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit
from pyspark.sql.types import DateType, BooleanType
from pyspark.sql import functions as F
from datetime import date

# Changed customer IDs
changed_ids = [id1, id2]
today = date.today()

# Load Delta table
dt = DeltaTable.forName(spark, "dim_customer")

# Step 1: Close current records
dt.update(
    condition=(
        col("customer_id").isin(changed_ids) &
        (col("is_current") == True)
    ),
    set={
        "end_date": lit(today).cast(DateType()),
        "is_current": lit(False).cast(BooleanType())
    }
)

# Step 2: Get existing customer records
existing = spark.table("dim_customer").filter(
    col("customer_id").isin(changed_ids)
)

# Find maximum customer_key
max_key = spark.table("dim_customer") \
    .agg(F.max("customer_key").alias("max_key")) \
    .collect()[0]["max_key"]

# Create new SCD Type 2 rows
new_rows = (
    existing
    .join(
        changes.select("customer_id", "new_segment"),
        on="customer_id",
        how="left"
    )
    .withColumn("segment", col("new_segment"))
    .withColumn("start_date", lit(today).cast(DateType()))
    .withColumn("end_date", lit(None).cast(DateType()))
    .withColumn("is_current", lit(True).cast(BooleanType()))
    .withColumn(
        "customer_key",
        F.monotonically_increasing_id() + max_key + 1
    )
    .drop("new_segment")
)

# Step 3: Append new records
new_rows.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("dim_customer")

print("SCD Type 2 applied successfully!")

# Display inserted rows
new_rows.show()


StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 26, Finished, Available, Finished, False)

SCD Type 2 applied successfully!
+-----------+------------+--------------+-----------+----------+--------+----------+
|customer_id|customer_key| customer_name|    segment|start_date|end_date|is_current|
+-----------+------------+--------------+-----------+----------+--------+----------+
|   LS-17200|         804|  Luke Schmidt|  Corporate|2026-07-02|    NULL|      true|
|   PN-18775|         805|Parhena Norris|Home Office|2026-07-02|    NULL|      true|
|   LS-17200|         806|  Luke Schmidt|  Corporate|2026-07-02|    NULL|      true|
|   PN-18775|         807|Parhena Norris|Home Office|2026-07-02|    NULL|      true|
+-----------+------------+--------------+-----------+----------+--------+----------+



In [14]:
# Cell S4 -- Verify SCD Type 2: both old and new rows exist for changed customers
spark.table('dim_customer') \
.filter(col('customer_id').isin(changed_ids)) \
.orderBy('customer_id','start_date') \
.show(truncate=False)

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 28, Finished, Available, Finished, False)

+------------+-----------+--------------+-----------+----------+----------+----------+
|customer_key|customer_id|customer_name |segment    |start_date|end_date  |is_current|
+------------+-----------+--------------+-----------+----------+----------+----------+
|804         |LS-17200   |Luke Schmidt  |Corporate  |2026-07-02|NULL      |true      |
|0           |LS-17200   |Luke Schmidt  |Corporate  |2026-07-02|2026-07-02|false     |
|805         |PN-18775   |Parhena Norris|Home Office|2026-07-02|NULL      |true      |
|1           |PN-18775   |Parhena Norris|Home Office|2026-07-02|2026-07-02|false     |
+------------+-----------+--------------+-----------+----------+----------+----------+



In [16]:
# Cell S5 -- Row count should be original count + 2

from pyspark.sql.functions import col

print(f"dim_customer total rows: {spark.table('dim_customer').count()}")

print(f"Current rows only: {spark.table('dim_customer').filter(col('is_current') == True).count()}")

StatementMeta(, b5b4c3ef-3108-4cb7-aa30-fec7ee095d2a, 30, Finished, Available, Finished, False)

dim_customer total rows: 806
Current rows only: 804
